In [4]:
import os
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. 從 src 資料夾匯入您剛寫好的兩個模組
from src.models.pytorch_model import DiabetesNet
from src.train_utils import train, evaluate

def load_local_parquet_data(group_name, batch_size=32):
    """讀取處理好的 Parquet 檔案並轉換為 DataLoader"""
    # 假設 Notebook 位於專案根目錄
    current_dir = os.getcwd()
    train_path = os.path.join(current_dir, 'data', 'processed', 'train', f'partition_{group_name}_train.parquet')
    test_path = os.path.join(current_dir, 'data', 'processed', 'test', f'partition_{group_name}_test.parquet')
    
    # 讀取 Parquet
    train_df = pd.read_parquet(train_path)
    test_df = pd.read_parquet(test_path)
    
    # 分離特徵與標籤
    X_train = train_df.drop(columns=['DIQ010']).values
    Y_train = train_df['DIQ010'].values
    
    X_test = test_df.drop(columns=['DIQ010']).values
    Y_test = test_df['DIQ010'].values
    
    # 轉換為 PyTorch Tensor，並針對 BCELoss 將 Y 擴充維度 (unsqueeze)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32).unsqueeze(1)
    
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32).unsqueeze(1)
    
    # 封裝為 DataLoader
    train_loader = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor), batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, X_train.shape[1]



In [11]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# 1. 建立 Dataset (100筆資料，特徵為5維)
x = torch.randn(100, 5)
y = torch.randint(0, 2, (100,))
dataset = TensorDataset(x, y)

# 2. 封裝進 DataLoader
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 3. 迭代取出的形式為 Tuple (特徵張量, 標籤張量)
for batch_x, batch_y in dataloader:
    print(batch_x.shape)  # 輸出: torch.Size([16, 5])
    print(batch_y.shape)  # 輸出: torch.Size([16])
    break

torch.Size([16, 5])
torch.Size([16])


In [17]:
train_loader, test_loader, X_train_shape = load_local_parquet_data("Group1", batch_size=32)

print(X_train_shape) # 輸出: 特徵維度數
for batch_x, batch_y in test_loader:
    print(batch_x.shape)  # 輸出: torch.Size([32, 16])
    print(batch_y.shape)  # 輸出: torch.Size([32, 1])
    break

16
torch.Size([32, 16])
torch.Size([32, 1])


In [ ]:
Groups = [ "Group1", "Group2", "Group3", "Group4", "Group5"]

for Group_name in Groups:
    




In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def train(model, dataloader, epochs, lr, device='cpu'):
    """
    訓練神經網路模型的核心函式。
    
    參數:
        model: 要訓練的 PyTorch 模型 (對應 DiabetesNet)
        dataloader: 提供訓練批次數據的 DataLoader
        epochs: 訓練的總輪數
        lr: 學習率 (Learning Rate)
        device: 執行運算的硬體 ('cpu' 或 'cuda')
        
    回傳:
        model: 訓練完成的模型
    """
    # 因 DiabetesNet 輸出為 1 維且有 Sigmoid，必須用二元交叉熵損失函數 (衡量模型預測的機率分佈與真實標籤（0 或 1）之間的差距)
    criterion = nn.BCELoss() 
    
    # 使用 Adam 優化器自動調整學習率並更新權重
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # 宣告模型進入訓練模式 (讓系統準備好計算梯度)
    model.train()
    
    for epoch in range(epochs):
        for batch_X, batch_y in dataloader:
            # 將特徵搬移至指定硬體
            batch_X = batch_X.to(device)
            
            # 且形狀必須與模型輸出一致 (batch_size, 1)，因此需使用 view(-1, 1) 擴充維度
            batch_y = batch_y.to(device).float().view(-1, 1)
            
            # 1. 清空上一輪的梯度
            optimizer.zero_grad()
            
            # 2. 前向傳播：讓模型給出預測機率 (0~1 之間)
            outputs = model(batch_X)
            
            # 3. 裁判計算損失 (Loss)
            loss = criterion(outputs, batch_y)
            
            # 4. 反向傳播：計算微積分梯度
            loss.backward()
            
            # 5. 教練根據梯度更新權重
            optimizer.step()
            
    return model

In [ ]:
# ==========================================
# 進行模組功能整合測試
# ==========================================
if __name__ == "__main__":
    # 2. 設定硬體
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用的運算硬體: {device}")
    
    # 3. 載入 Group1 的資料進行測試
    test_group = "Group1"
    train_loader, test_loader, input_dim = load_local_parquet_data(test_group)
    print(f"成功載入 {test_group} 資料！特徵維度: {input_dim}")
    
    # 4. 初始化模型
    model = DiabetesNet(input_dim=input_dim).to(device)
    
    # 5. 呼叫 train_utils.py 的 train 函式
    print("\n--- 開始模組化訓練 ---")
    trained_model = train(model, train_loader, epochs=30, lr=0.001, device=device)
    print("訓練完成！")
    
    # 6. 呼叫 train_utils.py 的 evaluate 函式
    print("\n--- 開始模組化評估 ---")
    loss, metrics = evaluate(trained_model, test_loader, device=device)
    
    print(f"測試集 Loss: {loss:.4f}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"F1-Score: {metrics['f1']:.4f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")